<a href="https://colab.research.google.com/github/ChitrashreeShankaranandha/AgenticAI_LLMs_RAG/blob/main/Saaay_Cheeeeeeesse!.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🧀 Project Overview: CheeseGPT**

CheeseGPT is an AI-powered assistant that helps users explore cheeses in an intelligent way. It combines machine learning and an agent-based system to:

- Predict the country of origin of a cheese based on its attributes

- Recommend cheese names based on user preferences

- Provide detailed information about specific cheeses

- Find similar cheeses based on characteristics

*The goal of this project is to show how structured data and machine learning can be used together to build an interactive, real-world AI application.*

# **⚙️ Step-by-Step Explanation of the Project**
1. Data Collection & Understanding

  * Used a Kaggle dataset containing information about cheeses - https://www.kaggle.com/datasets/joebeachcapital/cheese

  * Dataset includes features like:

    * Milk type (cow, goat, etc.)

    * Texture (soft, hard, etc.)

    * Flavor, aroma

    * Rind, color

    * Country of origin

👉 This is the foundation of the entire system

# **2. Data Cleaning & Preprocessing**

* Converted all column names to lowercase

* Filled missing values with "Unknown"

* Extracted a clean cheese name from the URL

* Selected relevant categorical features for modeling

👉 This ensures the data is consistent and usable

# **3. Feature Engineering**

* Converted categorical data into numerical format using DictVectorizer

* Selected important features: milk, texture, flavor, aroma, rind, etc.

👉 This step prepares the data for machine learning

# **4. Machine Learning Model**

Used **Logistic Regression** as the classification model

**Objective**: predict country of origin based on cheese attributes

Converted categorical features using DictVectorizer

Split dataset into training and testing sets

Evaluated using:

* Accuracy

* Precision / Recall / F1-score

👉 Final model achieved ~58% accuracy, which is reasonable given:

* Many classes (countries)

* Class imbalance in dataset

# **5. Handling Dataset Challenges**

Dataset had many countries with very few samples. This caused imbalance and poor predictions

**Solution:**

* Kept only meaningful classes

* Focused on improving feature quality instead of over-augmenting

👉 Shows understanding of real-world ML challenges

# **6. Building Intelligent Tools**

Instead of using only one model, the system includes multiple tools:

* Prediction Tool → ML model predicts country

* Profile Tool → Displays cheese details

* Filter Tool → Structured search

* Similarity Tool → Finds similar cheeses

* Recommendation Tool → Finds cheeses based on attributes

👉 Each tool solves a specific problem

# **7. Agent-Based System**

Built a simple AI agent loop

It:

- Uses rule-based logic to detect user intent
- Routes the query to the appropriate tool
- Executes the tool and retrieves results
- Passes results to the LLM for response generation

👉 Example:

- “Which cheese is soft and creamy?” → recommendation tool
- “Which country is this cheese from?” → prediction tool

# **8. LLM Response Layer**

Added an LLM layer to improve response quality and user interaction.

The LLM is used to:

- Convert tool outputs into natural, conversational responses
- Improve readability and formatting of results
- Ensure responses are clear, concise, and user-friendly
- Avoid hallucination by grounding responses in tool outputs

👉 The LLM does not perform prediction or decision-making

👉 It acts as a final response generation layer on top of ML + tools

# **9. User Interface (Gradio)**

Built an interactive chat UI using Gradio

Includes:

- Chat window

- Input box

- Agent trace panel (shows tool usage)

👉 Makes the system easy to use and explainable

In [ ]:
from google.colab import files
uploaded = files.upload()

In [2]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression

In [ ]:
# Load data
df = pd.read_csv("cheese_details.csv")
df.columns = [c.strip().lower() for c in df.columns]

# Features and target
categorical_features = [
    "milk", "region", "family", "type", "texture",
    "rind", "color", "flavor", "aroma",
    "vegetarian", "vegan"
]
target = "country"
link = "url"
name_col = "cheese_name"

# Keep only available columns
available_cat = [c for c in categorical_features if c in df.columns]
required_cols = available_cat + [target]
if name_col in df.columns:
    required_cols.append(name_col)

df = df[required_cols].copy()

# Fill missing values
for col in available_cat + [target]:
    df[col] = df[col].fillna("Unknown").astype(str)

# Remove empty target rows
df = df[df[target].str.strip().ne("")].copy()

# Remove rare countries
country_counts = df[target].value_counts()
valid_countries = country_counts[country_counts >= 5].index
df = df[df[target].isin(valid_countries)].copy()

print("Countries kept:\n", df[target].value_counts())

In [4]:
# Encode target AFTER filtering
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(df[target])

# Encode features
cat_records = df[available_cat].to_dict(orient="records")
dict_vec = DictVectorizer(sparse=True)
X = dict_vec.fit_transform(cat_records)

In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train model
model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

# Fit
model.fit(X_train, y_train)

In [ ]:
# Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print("Cross-validation accuracy:", cv_scores.mean())

In [ ]:
# Evaluate
y_pred = model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

In [ ]:
# Save artifacts
artifacts = {
    "model": model,
    "dict_vectorizer": dict_vec,
    "target_encoder": target_encoder,
    "categorical_features": available_cat,
    "target": target,
    "name_col": name_col if name_col in df.columns else None,
}

with open("cheese_artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("Saved cheese_artifacts.pkl")

In [9]:
import pickle
import pandas as pd

# **Load data and artifacts**

In [10]:
CSV_PATH = "cheese_details.csv"
ARTIFACTS_PATH = "cheese_artifacts.pkl"

df = pd.read_csv(CSV_PATH)
df.columns = [c.strip().lower() for c in df.columns]
df = df.fillna("Unknown")

# Create cheese_name from URL
df["cheese_name"] = (
    df["url"]
    .astype(str)
    .str.rstrip("/")
    .str.split("/")
    .str[-1]
    .str.replace("-", " ", regex=False)
    .str.title()
)

with open(ARTIFACTS_PATH, "rb") as f:
    artifacts = pickle.load(f)

model = artifacts["model"]
dict_vectorizer = artifacts["dict_vectorizer"]
target_encoder = artifacts["target_encoder"]
categorical_features = artifacts["categorical_features"]
target_col = artifacts["target"]

# **Helper functions**

In [11]:
def normalize_value(val):
    if val is None:
        return None
    val = str(val).strip()
    return val if val else None

***Tools***

In [12]:
def predict_origin_tool(
    milk=None, region=None, family=None, cheese_type=None,
    texture=None, rind=None, color=None, flavor=None,
    aroma=None, vegetarian=None, vegan=None
):
    record = {
        "milk": normalize_value(milk) or "Unknown",
        "region": normalize_value(region) or "Unknown",
        "family": normalize_value(family) or "Unknown",
        "type": normalize_value(cheese_type) or "Unknown",
        "texture": normalize_value(texture) or "Unknown",
        "rind": normalize_value(rind) or "Unknown",
        "color": normalize_value(color) or "Unknown",
        "flavor": normalize_value(flavor) or "Unknown",
        "aroma": normalize_value(aroma) or "Unknown",
        "vegetarian": normalize_value(vegetarian) or "Unknown",
        "vegan": normalize_value(vegan) or "Unknown",
    }

    # engineered features
    record["flavor_aroma"] = f"{record['flavor']} | {record['aroma']}"
    record["texture_rind"] = f"{record['texture']} | {record['rind']}"
    record["milk_type"] = f"{record['milk']} | {record['type']}"
    record["combined_profile"] = " | ".join([
        record["milk"],
        record["texture"],
        record["flavor"],
        record["aroma"],
        record["rind"],
        record["type"],
        record["color"]
    ])

    # keep only features used in training
    record = {k: v for k, v in record.items() if k in categorical_features}

    X_input = dict_vectorizer.transform([record])
    pred_idx = model.predict(X_input)[0]
    pred_country = target_encoder.inverse_transform([pred_idx])[0]

    top3 = []
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_input)[0]
        top_idx = probs.argsort()[-3:][::-1]
        for i in top_idx:
            top3.append({
                "country": target_encoder.inverse_transform([i])[0],
                "probability": round(float(probs[i]), 4)
            })

    return {
        "predicted_country": pred_country,
        "top3_predictions": top3,
        "input_used": record
    }

In [13]:
def filter_cheeses_tool(
    milk=None, country=None, region=None, family=None, cheese_type=None,
    texture=None, rind=None, color=None, flavor=None,
    aroma=None, vegetarian=None, vegan=None, producers=None, limit=10
):
    filtered = df.copy()

    criteria = {
        "milk": milk,
        "country": country,
        "region": region,
        "family": family,
        "type": cheese_type,
        "texture": texture,
        "rind": rind,
        "color": color,
        "flavor": flavor,
        "aroma": aroma,
        "vegetarian": vegetarian,
        "vegan": vegan,
        "producers": producers
    }

    for col, value in criteria.items():
        value = normalize_value(value)
        if value and col in filtered.columns:
            filtered = filtered[
                filtered[col].astype(str).str.contains(value, case=False, na=False)
            ]

    show_cols = [c for c in [
        "cheese_name", "url", "country", "region", "family", "type", "milk",
        "texture", "rind", "flavor", "aroma", "producers"
    ] if c in filtered.columns]

    return filtered[show_cols].head(limit).to_dict(orient="records")

In [14]:
import re
from difflib import get_close_matches

def extract_cheese_query(text):
    text = str(text).strip().lower()

    # remove common profile-style phrases
    patterns = [
        r"^tell me about\s+",
        r"^what is\s+",
        r"^what's\s+",
        r"^give me details about\s+",
        r"^details about\s+",
        r"^information about\s+",
        r"^info about\s+",
        r"^show me\s+",
    ]

    for p in patterns:
        text = re.sub(p, "", text)

    # remove generic trailing words
    text = re.sub(r"\bcheese\b", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [15]:
def get_profile_by_name_tool(name_text):
    query = extract_cheese_query(name_text)

    if not query:
        return {"error": "Please provide a cheese name."}

    # exact match first
    exact = df[df["cheese_name"].astype(str).str.lower() == query]
    if not exact.empty:
        return exact.iloc[0].to_dict()

    # substring in cheese_name
    matches = df[df["cheese_name"].astype(str).str.lower().str.contains(query, na=False)]
    if not matches.empty:
        return matches.iloc[0].to_dict()

    # synonyms
    if "synonyms" in df.columns:
        matches = df[df["synonyms"].astype(str).str.lower().str.contains(query, na=False)]
        if not matches.empty:
            return matches.iloc[0].to_dict()

    # alt spellings
    if "alt_spellings" in df.columns:
        matches = df[df["alt_spellings"].astype(str).str.lower().str.contains(query, na=False)]
        if not matches.empty:
            return matches.iloc[0].to_dict()

    # fuzzy fallback
    all_names = df["cheese_name"].astype(str).dropna().tolist()
    close = get_close_matches(query.title(), all_names, n=1, cutoff=0.6)
    if close:
        fuzzy = df[df["cheese_name"] == close[0]]
        if not fuzzy.empty:
            return fuzzy.iloc[0].to_dict()

    return {"error": f"No cheese entry found for '{query}'"}

In [16]:
def similar_cheeses_tool(
    name_text=None, milk=None, region=None, family=None, cheese_type=None,
    texture=None, rind=None, color=None, flavor=None, aroma=None, limit=5
):
    reference = None

    if name_text:
        matches = df[df["cheese_name"].astype(str).str.contains(str(name_text), case=False, na=False)]
        if matches.empty and "synonyms" in df.columns:
            matches = df[df["synonyms"].astype(str).str.contains(str(name_text), case=False, na=False)]
        if not matches.empty:
            reference = matches.iloc[0].to_dict()

    if reference is None:
        reference = {
            "milk": normalize_value(milk),
            "region": normalize_value(region),
            "family": normalize_value(family),
            "type": normalize_value(cheese_type),
            "texture": normalize_value(texture),
            "rind": normalize_value(rind),
            "color": normalize_value(color),
            "flavor": normalize_value(flavor),
            "aroma": normalize_value(aroma),
        }

    compare_cols = ["milk", "region", "family", "type", "texture", "rind", "color", "flavor", "aroma"]

    scored_rows = []
    for _, row in df.iterrows():
        score = 0
        for col in compare_cols:
            ref_val = reference.get(col)
            row_val = str(row[col]) if col in df.columns else None
            if ref_val and row_val and str(ref_val).lower() == str(row_val).lower():
                score += 1

        if score > 0:
            row_dict = row.to_dict()
            row_dict["similarity_score"] = score
            scored_rows.append(row_dict)

    scored_rows = sorted(scored_rows, key=lambda x: x["similarity_score"], reverse=True)

    show_results = []
    for row in scored_rows[:limit]:
        show_results.append({
            "cheese_name": row.get("cheese_name"),
            "url": row.get("url"),
            "country": row.get("country"),
            "type": row.get("type"),
            "milk": row.get("milk"),
            "texture": row.get("texture"),
            "rind": row.get("rind"),
            "flavor": row.get("flavor"),
            "aroma": row.get("aroma"),
            "producers": row.get("producers"),
            "similarity_score": row.get("similarity_score")
        })

    return show_results

In [17]:
def recommend_cheeses_tool(
    milk=None, region=None, family=None, cheese_type=None,
    texture=None, rind=None, color=None, flavor=None,
    aroma=None, vegetarian=None, vegan=None, limit=5
):
    filtered = df.copy()

    criteria = {
        "milk": milk,
        "region": region,
        "family": family,
        "type": cheese_type,
        "texture": texture,
        "rind": rind,
        "color": color,
        "flavor": flavor,
        "aroma": aroma,
        "vegetarian": vegetarian,
        "vegan": vegan
    }

    # Strict filtering first
    for col, value in criteria.items():
        value = normalize_value(value)
        if value and col in filtered.columns:
            filtered = filtered[
                filtered[col].astype(str).str.contains(value, case=False, na=False)
            ]

    if len(filtered) > 0:
        show_cols = [c for c in [
            "cheese_name", "url", "country", "type", "milk",
            "texture", "rind", "flavor", "aroma", "producers"
        ] if c in filtered.columns]
        return filtered[show_cols].head(limit).to_dict(orient="records")

    # Fallback similarity scoring
    compare_cols = ["milk", "region", "family", "type", "texture", "rind", "color", "flavor", "aroma", "vegetarian", "vegan"]

    scored_rows = []
    for _, row in df.iterrows():
        score = 0
        for col in compare_cols:
            value = criteria.get(col)
            if value and col in df.columns:
                if str(row[col]).lower() == str(value).lower():
                    score += 1

        if score > 0:
            row_dict = row.to_dict()
            row_dict["match_score"] = score
            scored_rows.append(row_dict)

    scored_rows = sorted(scored_rows, key=lambda x: x["match_score"], reverse=True)

    show_results = []
    for row in scored_rows[:limit]:
        show_results.append({
            "cheese_name": row.get("cheese_name"),
            "url": row.get("url"),
            "country": row.get("country"),
            "type": row.get("type"),
            "milk": row.get("milk"),
            "texture": row.get("texture"),
            "rind": row.get("rind"),
            "flavor": row.get("flavor"),
            "aroma": row.get("aroma"),
            "producers": row.get("producers"),
            "match_score": row.get("match_score")
        })

    return show_results

In [18]:
def summarize_results(results):
    if isinstance(results, dict):
        if "error" in results:
            return results["error"]

        # Prediction-style dict
        if "predicted_country" in results:
            text = f"Predicted country of origin: **{results['predicted_country']}**\n\n"
            if results.get("top3_predictions"):
                text += "Top predictions:\n"
                for pred in results["top3_predictions"]:
                    text += f"- {pred['country']} ({pred['probability']:.2%})\n"
            return text

        preferred_order = [
            "cheese_name", "url", "country", "region", "family", "type", "milk",
            "texture", "rind", "color", "flavor", "aroma",
            "vegetarian", "vegan", "producers", "synonyms", "alt_spellings"
        ]

        lines = []
        for key in preferred_order:
            if key in results:
                lines.append(f"**{key.replace('_', ' ').title()}**: {results[key]}")
        return "\n".join(lines)

    if isinstance(results, list):
        if not results:
            return "No matching cheeses found."

        lines = []
        for i, item in enumerate(results, 1):
            name = item.get("cheese_name", "Unknown")
            url = item.get("url", "")

            summary = f"**{i}. {name}**"
            if url:
                summary += f"\n🔗 {url}"

            details = []
            for key in ["country", "type", "milk", "texture", "rind", "flavor", "aroma", "producers", "similarity_score", "match_score"]:
                if key in item and item[key] not in [None, "", "Unknown"]:
                    details.append(f"{key.replace('_', ' ')}: {item[key]}")

            if details:
                summary += "\n" + " | ".join(details)

            lines.append(summary)

        return "\n\n".join(lines)

    return str(results)

# **Intent detection**

In [19]:
def detect_cheese_recommendation_request(message):
    message = message.lower()

    recommendation_keywords = [
        "which cheese",
        "what cheese",
        "find cheese",
        "find cheeses",
        "show cheeses",
        "recommend cheese",
        "recommend cheeses",
        "cheese made of",
        "made of"
    ]

    feature_keywords = [
        "milk", "texture", "flavor", "aroma", "rind", "soft", "hard", "washed", "creamy", "strong"
    ]

    return (
        any(k in message for k in recommendation_keywords)
        and any(k in message for k in feature_keywords)
    )

In [20]:
def detect_prediction_request(message):
    message = message.lower()

    prediction_keywords = [
        "predict origin",
        "country of origin",
        "which country",
        "likely from",
        "comes from",
        "origin of"
    ]

    feature_keywords = [
        "milk", "texture", "flavor", "aroma", "rind", "cheese"
    ]

    return (
        any(k in message for k in prediction_keywords)
        and any(k in message for k in feature_keywords)
    )

In [21]:
def detect_filter_request(message):
    words = ["find", "show", "list", "search", "filter"]
    return any(w in message.lower() for w in words)

In [22]:
def detect_profile_request(message):
    words = ["profile", "details", "tell me about", "what is", "information about"]
    return any(w in message.lower() for w in words)

In [23]:
def detect_similar_request(message):
    words = ["similar", "like this", "related", "closest"]
    return any(w in message.lower() for w in words)

In [24]:
def extract_known_values(message):
    extracted = {}

    for col in ["milk", "region", "family", "type", "texture", "rind", "color", "flavor", "aroma", "vegetarian", "vegan", "country"]:
        if col in df.columns:
            unique_vals = sorted(df[col].dropna().astype(str).unique(), key=len, reverse=True)
            for val in unique_vals:
                if val.lower() != "unknown" and val.lower() in message.lower():
                    extracted[col] = val
                    break

    return extracted

# **LLM setup**

In [ ]:
! pip install -q langchain-openai openai

import os
from langchain_openai import ChatOpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OpenAIApi')

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

def llm_format_answer(user_message, tool_name, raw_result_text):
    prompt = f"""
You are CheeseGPT, a helpful cheese assistant.

User message:
{user_message}

Tool used:
{tool_name}

Tool output:
{raw_result_text}

Instructions:
- Write a concise, natural, user-friendly answer.
- Do not invent facts that are not present in the tool output.
- If the tool output is empty, uncertain, or says the answer is unknown, say so clearly.
- Keep the response grounded in the provided tool output.
- Use short bullets when listing multiple cheeses.
"""

    response = llm.invoke(prompt)
    return response.content.strip()

# **Agent**

In [27]:
def run_agent(message):
    trace = []
    extracted = extract_known_values(message)

    # 1. Cheese recommendation by attributes
    if detect_cheese_recommendation_request(message):
        trace.append("Tool used: recommend_cheeses_tool")
        result = recommend_cheeses_tool(
            milk=extracted.get("milk"),
            region=extracted.get("region"),
            family=extracted.get("family"),
            cheese_type=extracted.get("type"),
            texture=extracted.get("texture"),
            rind=extracted.get("rind"),
            color=extracted.get("color"),
            flavor=extracted.get("flavor"),
            aroma=extracted.get("aroma"),
            vegetarian=extracted.get("vegetarian"),
            vegan=extracted.get("vegan"),
            limit=5
        )
        raw_text = summarize_results(result)
        final_answer = llm_format_answer(message, "recommend_cheeses_tool", raw_text)
        return final_answer, "\n".join(trace)

    # 2. Country prediction
    if detect_prediction_request(message):
        trace.append("Tool used: predict_origin_tool")
        result = predict_origin_tool(
            milk=extracted.get("milk"),
            region=extracted.get("region"),
            family=extracted.get("family"),
            cheese_type=extracted.get("type"),
            texture=extracted.get("texture"),
            rind=extracted.get("rind"),
            color=extracted.get("color"),
            flavor=extracted.get("flavor"),
            aroma=extracted.get("aroma"),
            vegetarian=extracted.get("vegetarian"),
            vegan=extracted.get("vegan")
        )
        raw_text = summarize_results(result)
        final_answer = llm_format_answer(message, "predict_origin_tool", raw_text)
        return final_answer, "\n".join(trace)

    # 3. Similar cheeses
    if detect_similar_request(message):
        trace.append("Tool used: similar_cheeses_tool")
        result = similar_cheeses_tool(
            name_text=message,
            milk=extracted.get("milk"),
            region=extracted.get("region"),
            family=extracted.get("family"),
            cheese_type=extracted.get("type"),
            texture=extracted.get("texture"),
            rind=extracted.get("rind"),
            color=extracted.get("color"),
            flavor=extracted.get("flavor"),
            aroma=extracted.get("aroma"),
            limit=5
        )
        raw_text = summarize_results(result)
        final_answer = llm_format_answer(message, "similar_cheeses_tool", raw_text)
        return final_answer, "\n".join(trace)

    # 4. Cheese profile
    if detect_profile_request(message):
        trace.append("Tool used: get_profile_by_name_tool")
        result = get_profile_by_name_tool(message)
        raw_text = summarize_results(result)
        final_answer = llm_format_answer(message, "get_profile_by_name_tool", raw_text)
        return final_answer, "\n".join(trace)

    # 5. Structured filtering
    if detect_filter_request(message):
        trace.append("Tool used: filter_cheeses_tool")
        result = filter_cheeses_tool(
            milk=extracted.get("milk"),
            country=extracted.get("country"),
            region=extracted.get("region"),
            family=extracted.get("family"),
            cheese_type=extracted.get("type"),
            texture=extracted.get("texture"),
            rind=extracted.get("rind"),
            color=extracted.get("color"),
            flavor=extracted.get("flavor"),
            aroma=extracted.get("aroma"),
            vegetarian=extracted.get("vegetarian"),
            vegan=extracted.get("vegan"),
            limit=10
        )
        raw_text = summarize_results(result)
        final_answer = llm_format_answer(message, "filter_cheeses_tool", raw_text)
        return final_answer, "\n".join(trace)

    # Fallback
    trace.append("Fallback triggered")
    fallback_text = (
        "I don’t know the answer to that yet. "
        "You can ask about cheese origin, cheese profiles, similar cheeses, "
        "or request recommendations based on attributes."
    )
    final_answer = llm_format_answer(message, "fallback", fallback_text)
    return final_answer, "\n".join(trace)

# **Gradio UI**

In [ ]:
import gradio as gr

def chat_fn(message, history):
    if not message.strip():
        return history, "", ""

    answer, trace = run_agent(message)
    print(f"LLM response is: {answer}")
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer}
    ]
    return history, trace, ""

def load_example(example_text, history):
    answer, trace = run_agent(example_text)
    history = history + [
        {"role": "user", "content": example_text},
        {"role": "assistant", "content": answer}
    ]
    return history, trace, ""

def clear_all():
    return [], "", ""

custom_css = """
.gradio-container {
    max-width: 1400px !important;
}
.hero {
    text-align: center;
    padding: 10px 0 20px 0;
}
.hero h1 {
    margin-bottom: 6px;
}
.hero p {
    font-size: 17px;
    color: #555;
}
.section-card {
    border-radius: 14px;
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:
    with gr.Column(elem_classes="hero"):
        gr.Markdown(
            """
            # 🧀 CheeseGPT
            ### From Milk to Mystery: An Intelligent Cheese Discovery System
            <p>
            CheeseGPT combines <b>machine learning</b> and an <b>agent-based system</b> to recommend cheeses,
            predict origin country, show cheese profiles, and find similar cheeses.
            </p>
            """
        )

    with gr.Row():
        with gr.Column(scale=3):
            gr.Markdown("### 💬 Cheese Assistant")

            chatbot = gr.Chatbot(
                type="messages",
                label=None,
                height=520,
                show_copy_button=True,
                bubble_full_width=False
            )

            with gr.Row():
                msg = gr.Textbox(
                    label="Ask something",
                    placeholder="Try: Which cheese is made of soft cow milk with strong aroma and washed rind?",
                    lines=2,
                    scale=8
                )
                send_btn = gr.Button("Send", variant="primary", scale=1)

            with gr.Row():
                clear_btn = gr.Button("Clear Chat")
                ex1 = gr.Button("Recommend a cheese")

        with gr.Column(scale=2):
            gr.Markdown("### 🧠 Agent Trace")
            trace_box = gr.Textbox(
                label=None,
                lines=4,
                max_lines=100,
                interactive=False,
                placeholder="The selected tool and reasoning trace will appear here..."
            )

            gr.Markdown("### ℹ️ What this system can do")
            gr.Markdown(
                """
                - Recommend cheese names from attributes
                - Predict country of origin
                - Show cheese profiles
                - Find similar cheeses
                """
            )

            gr.Markdown(
                """
                **Example prompts**
                - Which cheese is made of soft cow milk with strong aroma and washed rind?
                - Which country is a soft cow milk cheese with washed rind likely from?
                - Tell me about Cheddar
                - Find cheeses similar to Brie
                """
            )


    send_btn.click(chat_fn, inputs=[msg, chatbot], outputs=[chatbot, trace_box, msg])
    msg.submit(chat_fn, inputs=[msg, chatbot], outputs=[chatbot, trace_box, msg])

    clear_btn.click(clear_all, outputs=[chatbot, trace_box, msg])

    ex1.click(
        fn=lambda h: load_example(
            "Which cheese is made of soft cow milk with strong aroma and washed rind?",
            h
        ),
        inputs=[chatbot],
        outputs=[chatbot, trace_box, msg]
    )

demo.launch(debug=True)